# Entregável 7 — Engenharia de Features

**Disciplina:** Aquisição e Processamento de Biossinais  
**Equipe:** José Ferreira Lessa & Matheus Rocha Gomes da Silva  
**Orientador:** Prof. Dr. Victor Hugo C. de Albuquerque  
**Dataset:** PTB-XL — A Large Publicly Available Electrocardiography Dataset (PhysioNet)  
**Referência:** Wagner et al. (2020). PTB-XL, a large publicly available electrocardiography dataset. *Scientific Data*, 7(1), 154.  
**Data:** Abril e Maio de 2026

---

## Objetivo

Este notebook realiza a **Engenharia de Features** sobre o dataset bruto gerado no Entregável 6 (`features_raw.parquet`), preparando um vetor de atributos limpo, enriquecido e normalizado para a etapa de Redução de Dimensionalidade (Entregável 8).

A engenharia de features atua em quatro frentes complementares:

1. **Diagnóstico e limpeza analítica:** remoção de features constantes e imputação de valores ausentes com proteção estrita contra data leakage.
2. **Features derivadas de segunda ordem:** construção de razões espectrais, normalizações por baseline e deltas morfológicos inter-derivações — combinações com interpretação fisiológica direta que ampliam a discriminabilidade entre classes.
3. **Normalização robusta:** escalonamento via `RobustScaler` fitado exclusivamente no conjunto de treino (folds 1–8), tornando o vetor de features invariante a outliers residuais e a diferenças de escala entre domínios.
4. **Validação de relevância:** análise de correlação das features com a variável resposta (ANOVA F-statistic, one-vs-rest) e análise de redundância entre pares de features, antecipando a seleção formal do Entregável 9.

O produto final é o arquivo `features_engineered.parquet`, que constitui a entrada direta do Entregável 8.

## 1. Importações, Configurações e Dependências

Todas as bibliotecas utilizadas neste notebook já foram apresentadas em entregáveis anteriores. A única adição relevante é o módulo `sklearn.preprocessing.RobustScaler`, descrito abaixo.

**`RobustScaler` (scikit-learn):** escalonador que centraliza cada feature pela mediana e divide pelo IQR (intervalo interquartil, P75 − P25), em vez de usar média e desvio padrão como o `StandardScaler`. Essa escolha é mais adequada aqui porque o dataset de features contém distribuições assimétricas e outliers residuais — especialmente nas features não-lineares — e o uso de média/desvio padrão distorceria o escalonamento para a maioria das amostras.

In [ ]:
import os
import ast
import gc
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from pathlib import Path
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import f_classif
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

np.random.seed(42)

## 2. Configurações Globais e Carregamento

In [ ]:
LEAD_NAMES   = ['I', 'II', 'III', 'aVL', 'aVR', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
FOLDS_TREINO = [1, 2, 3, 4, 5, 6, 7, 8]
FOLD_VAL     = 9
FOLD_TEST    = 10

DIR_IN_D6 = Path('../../entregavel-6/outputs/')
FIGS_DIR  = Path('../figuras/')
OUT_DIR   = Path('../outputs/')

for d in [FIGS_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Colunas de metadados — não são features, nunca entram no scaler
META_COLS = ['patient_id', 'strat_fold', 'sqi_category',
             'diagnostic_superclass', 'label_primary',
             'superclasses_clean', 'n_superclasses', 'split']

print('Configuração concluída.')
print(f'Figuras em : {FIGS_DIR.resolve()}')
print(f'Outputs em : {OUT_DIR.resolve()}')

In [ ]:
parquet_path = DIR_IN_D6 / 'features_raw.parquet'

if not parquet_path.exists():
    raise FileNotFoundError(
        f'Arquivo não encontrado: {parquet_path}\n'
        'Execute o Entregável 6 antes de prosseguir.'
    )

print('Carregando features brutas do Entregável 6...')
df = pd.read_parquet(str(parquet_path))

# Reconversão da coluna de superclasses (armazenada como string no parquet)
if isinstance(df['diagnostic_superclass'].iloc[0], str):
    df['diagnostic_superclass'] = df['diagnostic_superclass'].apply(ast.literal_eval)

# Label primário para agrupamentos e plots
if 'label_primary' not in df.columns:
    df['label_primary'] = df['diagnostic_superclass'].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) > 0 else 'UNKNOWN'
    )

# Atualiza META_COLS para remover colunas que podem não existir no parquet
META_COLS = [c for c in META_COLS if c in df.columns]

feature_cols = [c for c in df.columns if c not in META_COLS]

print(f'Dimensão inicial : {df.shape}')
print(f'Features brutas  : {len(feature_cols)}')
print(f'Registros        : {len(df)}')
print(f'\nDistribuição por fold:')
display(df['strat_fold'].value_counts().sort_index().to_frame('registros').T)

---

## Seção 1 — Diagnóstico e Limpeza Analítica do Dataset

### 1.1 Fundamentação

Antes de construir qualquer feature derivada ou normalizar o dataset, é necessário garantir que o conjunto de atributos herdado do Entregável 6 não contenha:

- **Features constantes ou quasi-constantes** (variância próxima de zero): features sem variação não carregam nenhuma informação discriminativa e podem causar instabilidade numérica no `RobustScaler` (divisão por IQR ≈ 0) e em algoritmos de redução de dimensionalidade como o PCA.

- **Features com alta taxa de valores ausentes** (NaN > 5%): como documentado no Entregável 6, as features não-lineares baseadas na série RR (`nonlin_sampen_rr`, `nonlin_sd1`, etc.) produzem NaN para registros com menos de 8 batimentos detectados. Features com mais de 5% de ausência são eliminadas por comprometerem a representatividade do dado; as demais são imputadas.

- **Valores ausentes residuais**: os NaNs remanescentes (taxa < 5%) são imputados pela **mediana da superclasse correspondente**, calculada exclusivamente sobre o conjunto de treino (folds 1–8). Essa estratégia é preferível à imputação por mediana global porque:
  - Respeita as diferenças fisiológicas entre classes (ex: a mediana de RMSSD de pacientes normais é diferente da de pacientes com MI).
  - Não vaza informação dos conjuntos de validação (fold 9) e teste (fold 10) para a estimativa dos parâmetros de imputação.

### 1.2 Remoção de Features Constantes e de Alta Taxa de NaN

In [ ]:
feature_cols = [c for c in df.columns if c not in META_COLS]
print(f'Features antes da limpeza: {len(feature_cols)}')

# ── 1.2a. Remoção de features quasi-constantes (variância < 1e-6) ────────────
vars_feat = df[feature_cols].var()
cols_zero_var = vars_feat[vars_feat < 1e-6].index.tolist()

if cols_zero_var:
    print(f'\nRemovendo {len(cols_zero_var)} features quasi-constantes (var < 1e-6):')
    for c in cols_zero_var:
        print(f'  - {c}  (var = {vars_feat[c]:.2e})')
    df.drop(columns=cols_zero_var, inplace=True)
else:
    print('\nNenhuma feature quasi-constante encontrada.')

feature_cols = [c for c in df.columns if c not in META_COLS]

# ── 1.2b. Remoção de features com NaN > 5% ──────────────────────────────────
null_pct = df[feature_cols].isnull().mean()
cols_high_nan = null_pct[null_pct > 0.05].index.tolist()

if cols_high_nan:
    print(f'\nRemovendo {len(cols_high_nan)} features com NaN > 5%:')
    for c in cols_high_nan:
        print(f'  - {c}  ({null_pct[c]*100:.1f}% ausente)')
    df.drop(columns=cols_high_nan, inplace=True)
else:
    print('\nNenhuma feature com NaN > 5% encontrada.')

feature_cols = [c for c in df.columns if c not in META_COLS]
features_com_nan = [c for c in feature_cols if df[c].isnull().any()]

print(f'\nFeatures após remoção  : {len(feature_cols)}')
print(f'Features com NaN restante (< 5%): {len(features_com_nan)}')

### 1.3 Imputação por Mediana da Superclasse (Treino-Restrita)

A imputação é realizada exclusivamente com base na mediana calculada no conjunto de treino, preservando o isolamento dos folds de validação e teste.

In [ ]:
mask_treino = df['strat_fold'].isin(FOLDS_TREINO)
df_treino   = df[mask_treino]

# Dicionário: {feature → {classe → mediana_treino, '_global' → mediana_global_treino}}
median_map = {}
for feat in features_com_nan:
    med_por_classe  = df_treino.groupby('label_primary')[feat].median().to_dict()
    med_global      = df_treino[feat].median()
    median_map[feat] = (med_por_classe, med_global)

# Imputação segura registro a registro
n_imputados = 0
for feat in features_com_nan:
    med_por_classe, med_global = median_map[feat]
    idx_null = df.index[df[feat].isnull()]

    for idx in idx_null:
        classe   = df.loc[idx, 'label_primary']
        valor    = med_por_classe.get(classe, med_global)
        if pd.isna(valor):
            valor = med_global
        df.loc[idx, feat] = valor
        n_imputados += 1

nulos_final = df[feature_cols].isnull().sum().sum()
print(f'Valores imputados  : {n_imputados}')
print(f'Nulos pós-imputação: {nulos_final}  (esperado: 0)')
assert nulos_final == 0, 'Ainda existem NaNs após imputação — revisar.'
print('\nDataset sem valores ausentes. Pronto para engenharia de features.')

---

## Seção 2 — Features Derivadas de Segunda Ordem

### 2.1 Fundamentação

O Entregável 6 produziu atributos de **primeira ordem** — cada feature mede uma propriedade do sinal de forma independente e absoluta (ex: potência da banda QRS em DII, amplitude R mediana em V5). A engenharia de features de segunda ordem constrói **combinações** dessas grandezas que, além de reduzir a dependência de escala absoluta, têm interpretação fisiológica direta.

O pipeline do curso prevê três estratégias para este entregável:

| Estratégia | Justificativa |
|---|---|
| **Razões entre bandas espectrais** | Razões são adimensionais e invariantes à amplitude absoluta do sinal — dois pacientes com ECG de amplitudes diferentes mas morfologia similar terão a mesma razão. |
| **Normalização por baseline (NORM)** | Expressa o afastamento de cada registro em relação ao comportamento médio de sinais normais, tornando a feature diretamente interpretável como "quão anormal é este valor". |
| **Deltas morfológicos inter-derivações** | A diferença de amplitude entre derivações anatomicamente complementares captura gradientes elétricos cardíacos que features individuais por derivação não conseguem representar. |

Todas as features derivadas são calculadas sobre **todo o dataset**, mas os parâmetros de referência (medianas de NORM, limiares) são extraídos exclusivamente do conjunto de treino.

### 2.2 Razões Espectrais (QRS/PT e QRS/Total)

In [ ]:
new_feats = {}

# ── A. Razão QRS/PT por derivação ────────────────────────────────────────────
# Fundamento: a relação entre a energia do complexo QRS e das ondas lentas (P+T)
# é sensível a alterações de condução (CD aumenta PT relativo ao QRS) e hipertrofia
# (HYP eleva QRS de forma desproporcionada). Por ser uma razão, é invariante à
# amplitude absoluta do sinal, tornando-a mais robusta a diferenças de impedância.
for lead in LEAD_NAMES:
    col_qrs = f'freq_qrs_power_{lead}'
    col_pt  = f'freq_pt_power_{lead}'
    if col_qrs in df.columns and col_pt in df.columns:
        new_feats[f'ratio_qrs_pt_{lead}'] = (
            df[col_qrs] / (df[col_pt] + 1e-10)
        )

# ── B. Razão QRS/Total por derivação ─────────────────────────────────────────
# Complementar à razão QRS/PT: expressa a fração da energia total que é explicada
# pelo complexo QRS. Em bloqueios de ramo, o QRS é mais largo e tem energia
# distribuída para frequências mais baixas, reduzindo essa razão.
for lead in LEAD_NAMES:
    col_qrs = f'freq_qrs_power_{lead}'
    col_tot = f'freq_total_power_{lead}'
    if col_qrs in df.columns and col_tot in df.columns:
        new_feats[f'ratio_qrs_total_{lead}'] = (
            df[col_qrs] / (df[col_tot] + 1e-10)
        )

df_new = pd.DataFrame(new_feats, index=df.index)
df     = pd.concat([df, df_new], axis=1)
del df_new, new_feats
gc.collect()

n_ratio = sum(1 for c in df.columns if c.startswith('ratio_'))
print(f'Features de razão espectral adicionadas: {n_ratio}')

### 2.3 Normalização por Baseline NORM

Cada feature de amplitude é expressa como múltiplo da mediana do grupo NORM no conjunto de treino. Um valor de 1,0 indica comportamento equivalente ao normal; valores acima ou abaixo indicam o grau de desvio.

In [ ]:
new_feats = {}
mask_norm_treino = mask_treino & (df['label_primary'] == 'NORM')

# Features de amplitude sobre as quais a normalização por baseline faz sentido:
# RMS e MAV são medidas de energia — normalizá-las pelo baseline NORM transforma
# a feature em "fator de amplitude relativo ao normal", interpretável clinicamente.
amplitude_prefixes = ['time_rms_', 'time_mav_']

for prefix in amplitude_prefixes:
    for lead in LEAD_NAMES:
        col = f'{prefix}{lead}'
        if col not in df.columns:
            continue

        med_norm = df.loc[mask_norm_treino, col].median()
        if pd.isna(med_norm) or med_norm == 0:
            continue

        nome_novo = col.replace('time_', 'norm_baseline_')
        new_feats[nome_novo] = df[col] / med_norm

df_new = pd.DataFrame(new_feats, index=df.index)
df     = pd.concat([df, df_new], axis=1)
del df_new, new_feats
gc.collect()

n_norm_bl = sum(1 for c in df.columns if c.startswith('norm_baseline_'))
print(f'Features de normalização por baseline adicionadas: {n_norm_bl}')

### 2.4 Deltas Morfológicos Inter-Derivações

A diferença de amplitude R entre derivações anatomicamente opostas (ex: DI e aVR, que são eletricamente complementares) representa o **gradiente elétrico** entre regiões cardíacas distintas. Essa informação não está presente em nenhuma feature individual de derivação.

In [ ]:
new_feats = {}

# ── A. Delta R entre derivações complementares ───────────────────────────────
# Pares com complementaridade anatômica estabelecida na literatura:
# - DI vs aVR: eixo horizontal esquerdo-direito (opostos elétricos)
# - DII vs aVL: eixo oblíquo inferior-lateral
# - V1 vs V6: gradiente transversal septo-lateral
pares_delta = [
    ('I', 'aVR'),
    ('II', 'aVL'),
    ('V1', 'V6'),
    ('V2', 'V5'),
]

for l1, l2 in pares_delta:
    col1 = f'morph_r_amp_{l1}_median' if f'morph_r_amp_{l1}_median' in df.columns else f'time_rms_{l1}'
    col2 = f'morph_r_amp_{l2}_median' if f'morph_r_amp_{l2}_median' in df.columns else f'time_rms_{l2}'
    if col1 in df.columns and col2 in df.columns:
        new_feats[f'delta_amp_{l1}_{l2}'] = df[col1] - df[col2]

# ── B. Delta de energia wavelet entre níveis adjacentes (D3 - D4) ────────────
# Captura a concentração relativa de energia entre a faixa do pico QRS (D3: 12.5-25 Hz)
# e a faixa de início do QRS (D4: 6.25-12.5 Hz). Em bloqueios de ramo, essa diferença
# tende a diminuir (energia mais distribuída entre os dois níveis).
if 'wavelet_energy_D3' in df.columns and 'wavelet_energy_D4' in df.columns:
    new_feats['delta_wavelet_D3_D4'] = df['wavelet_energy_D3'] - df['wavelet_energy_D4']

# ── C. Razão SD1/SD2 já extraída no E6 como feature não-linear ──────────────
# Não duplicamos — já está em nonlin_sd1_sd2_ratio

df_new = pd.DataFrame(new_feats, index=df.index)
df     = pd.concat([df, df_new], axis=1)
del df_new, new_feats
gc.collect()

feature_cols = [c for c in df.columns if c not in META_COLS]
n_derivadas  = sum(1 for c in feature_cols
                   if c.startswith(('ratio_', 'norm_baseline_', 'delta_')))

print(f'Features derivadas totais adicionadas: {n_derivadas}')
print(f'Total de features no dataset         : {len(feature_cols)}')

### 2.5 Resumo das Features Adicionadas

In [ ]:
grupos = {
    'Razões Espectrais (ratio_qrs_pt / ratio_qrs_total)':
        [c for c in feature_cols if c.startswith('ratio_')],
    'Normalização por Baseline NORM (norm_baseline_)':
        [c for c in feature_cols if c.startswith('norm_baseline_')],
    'Deltas Inter-Derivações / Wavelet (delta_)':
        [c for c in feature_cols if c.startswith('delta_')],
}

rows = []
for grupo, cols in grupos.items():
    rows.append({'Grupo': grupo, 'Nº de Features': len(cols), 'Exemplos': ', '.join(cols[:3])})

display(pd.DataFrame(rows).set_index('Grupo'))
print(f'\nTotal de features originais + derivadas: {len(feature_cols)}')

---

## Seção 3 — Normalização Robusta (RobustScaler)

### 3.1 Fundamentação

O dataset de features reúne atributos de domínios completamente distintos, com ordens de grandeza muito diferentes entre si:

- Features temporais (RMS, MAV): tipicamente na faixa de 0,01 – 2,0 mV
- Potência espectral (Welch): pode variar de 1e-6 a 1,0 mV²/Hz
- HRV (meanRR, RMSSD): centenas de milissegundos
- Features não-lineares (DFA, SampEn): adimensionais, entre 0 e 3

Sem normalização, algoritmos sensíveis à escala — como SVM com kernel RBF, kNN e PCA — são dominados pelas features de maior magnitude, independentemente de sua relevância diagnóstica. A normalização resolve esse problema colocando todas as features na mesma escala.

**Por que RobustScaler em vez de StandardScaler?**

O `StandardScaler` usa média e desvio padrão, que são estimadores não-robustos — um único outlier expressivo pode distorcer o centro e a escala estimados. O `RobustScaler` usa mediana e IQR:

$$z_i = \frac{x_i - Q_{50}}{Q_{75} - Q_{25}}$$

Isso garante que a escala seja determinada pelos 50% centrais da distribuição, tornando-a insensível a valores extremos residuais — exatamente o cenário presente neste dataset após a extração de features não-lineares.

### 3.2 Regra de Ouro: Fit Apenas no Treino

O scaler é **fitado exclusivamente nos folds 1–8** (conjunto de treino) e depois **aplicado** (transform) a todo o dataset — treino, validação (fold 9) e teste (fold 10). Isso garante que os parâmetros de escala (mediana e IQR de cada feature) não sejam contaminados por informações dos conjuntos de avaliação.

In [ ]:
feature_cols = [c for c in df.columns if c not in META_COLS]

X_treino = df.loc[mask_treino, feature_cols].values
X_todos  = df[feature_cols].values

print(f'Ajustando RobustScaler nos folds 1–8 ({mask_treino.sum()} registros)...')
scaler = RobustScaler()
scaler.fit(X_treino)

X_scaled = scaler.transform(X_todos)

# Reconstrução do DataFrame com metadados preservados
df_scaled = pd.DataFrame(X_scaled, columns=feature_cols, index=df.index)
for col in META_COLS:
    df_scaled[col] = df[col]

# Serialização do scaler para reprodutibilidade nos entregáveis seguintes
joblib.dump(scaler, OUT_DIR / 'scaler_robust.pkl')
print('RobustScaler salvo em scaler_robust.pkl')
print(f'\nDimensão do dataset escalado: {df_scaled.shape}')

### 3.3 Validação da Normalização

Após o escalonamento, verificamos se as features apresentam as propriedades esperadas: mediana próxima de 0 e IQR próximo de 1 no conjunto de treino (por definição do RobustScaler), com dispersão moderada fora desse intervalo.

In [ ]:
# Verificação estatística no conjunto de treino
df_tr_scaled = df_scaled[df_scaled['strat_fold'].isin(FOLDS_TREINO)]

medians_pos = df_tr_scaled[feature_cols].median()
iqrs_pos    = df_tr_scaled[feature_cols].quantile(0.75) - df_tr_scaled[feature_cols].quantile(0.25)

print('Verificação pós-escalonamento (conjunto de treino):')
print(f'  Mediana das medianas : {medians_pos.median():.4f}  (esperado ≈ 0.0)')
print(f'  Mediana dos IQRs     : {iqrs_pos.median():.4f}  (esperado ≈ 1.0)')
print(f'  Features com |mediana| > 0.1 : {(medians_pos.abs() > 0.1).sum()}')
print(f'  Features com IQR fora [0.8, 1.2]: {((iqrs_pos < 0.8) | (iqrs_pos > 1.2)).sum()}')

In [ ]:
# Distribuição de 6 features representativas antes vs depois da normalização
amostra_feats = []
for prefix in ['time_rms_II', 'freq_qrs_power_II', 'hrv_rmssd',
               'wavelet_energy_D3', 'nonlin_higuchi_fd', 'ratio_qrs_pt_II']:
    if prefix in feature_cols:
        amostra_feats.append(prefix)

fig, axes = plt.subplots(2, len(amostra_feats), figsize=(4 * len(amostra_feats), 8))

for j, feat in enumerate(amostra_feats):
    # Antes
    ax_antes = axes[0, j]
    ax_antes.hist(df.loc[mask_treino, feat].dropna(), bins=60,
                  color='steelblue', edgecolor='none', alpha=0.8)
    ax_antes.set_title(f'{feat}\n(original)', fontsize=8)
    ax_antes.set_ylabel('Contagem' if j == 0 else '')

    # Depois
    ax_dep = axes[1, j]
    ax_dep.hist(df_scaled.loc[mask_treino, feat].dropna(), bins=60,
                color='darkorange', edgecolor='none', alpha=0.8)
    ax_dep.set_title('(escalada)', fontsize=8)
    ax_dep.set_ylabel('Contagem' if j == 0 else '')
    ax_dep.axvline(0, color='black', lw=0.8, ls='--')

axes[0, 0].set_ylabel('Antes\nContagem')
axes[1, 0].set_ylabel('Depois\nContagem')

plt.suptitle('Distribuição de Features Selecionadas — Antes vs. Depois do RobustScaler\n(conjunto de treino)',
             fontsize=12)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'normalizacao_antes_depois.png', dpi=150, bbox_inches='tight')
plt.show()

**Comentários sobre a Seção 3.3 — Distribuições antes vs. depois do RobustScaler:**

> *Preencha esta célula após executar o bloco acima. Orientações específicas para o que analisar:*

- **`time_rms_II` (amplitude temporal):** antes do escalonamento, essa feature deve apresentar distribuição com cauda direita longa — alguns registros com hipertrofia ventricular (HYP) têm RMS muito acima da média. Após o RobustScaler, essa cauda deve ser comprimida mas ainda visível. Se a distribuição pós-escalonamento ainda tiver valores acima de +5 em muitos registros, considere aplicar transformação logarítmica antes de escalonar no Entregável 9.

- **`freq_qrs_power_II` (potência espectral):** a potência espectral bruta tem distribuição naturalmente log-normal — a maioria dos registros tem valores baixos e poucos têm valores muito altos. O RobustScaler não corrige a assimetria log-normal, apenas reduz a escala. Comente se a distribuição após o escalonamento ainda é visivelmente assimétrica — isso é esperado e não é problema para o PCA, mas pode influenciar classificadores sensíveis à distribuição no Entregável 8.

- **`hrv_rmssd` (variabilidade de curto prazo):** o RMSSD é fortemente influenciado por outliers de batimento — um único intervalo RR anômalo (ex: batimento ectópico não removido) pode inflar o RMSSD de um registro. Verifique se, após o escalonamento, ainda existem valores extremos positivos expressivos. Se sim, isso indica que a detecção Pan-Tompkins do Entregável 5 deixou batimentos anômalos em alguns registros.

- **`wavelet_energy_D3` (energia da banda QRS na wavelet):** esta feature deve apresentar, após o escalonamento, uma distribuição mais compacta que a potência espectral bruta — porque a DWT é calculada sobre o sinal já filtrado e winsorizados (Entregável 4), o que limita valores extremos. Se a distribuição pós-escalonamento for bastante simétrica, isso valida o pipeline de limpeza do Entregável 4.

- **`nonlin_higuchi_fd` (dimensão fractal):** por construção matemática, a dimensão fractal de Higuchi está confinada ao intervalo [1, 2]. A distribuição original já é bem comportada, e o RobustScaler deve produzir uma distribuição pós-escalonamento compacta e próxima do normal — possivelmente a mais simétrica entre todas as features mostradas. Discuta se isso ocorre.

- **`ratio_qrs_pt_II` (razão espectral — feature derivada):** esta é uma feature criada neste entregável. Compare sua distribuição original com a das potências individuais (`freq_qrs_power_II` e `freq_pt_power_II`) — a razão deve ter amplitude de variação menor e distribuição mais concentrada, confirmando que features relativas são naturalmente mais estáveis. Após o escalonamento, discuta se ela está comparável em escala às demais features.

---

## Seção 4 — Validação de Relevância e Redundância

### 4.1 Fundamentação

Com o dataset normalizado, realizamos duas análises de validação obrigatórias previstas no pipeline:

1. **Correlação com a variável resposta (ANOVA F-statistic):** avalia o poder discriminativo de cada feature individualmente, identificando quais atributos mais separam as classes diagnósticas.

2. **Análise de redundância (correlação de Pearson entre features):** identifica pares de features altamente correlacionadas (|r| > 0,90) que carregam informação essencialmente duplicada, antecipando as decisões de seleção do Entregável 9.

> **Nota importante:** este entregável realiza apenas a *identificação* da redundância — a remoção formal de features será realizada no Entregável 9 (Seleção de Atributos), onde métodos filter, wrapper e embedded são aplicados com validação estatística rigorosa.

### 4.2 ANOVA F-Statistic — Relevância por Classe (One-vs-Rest)

O teste ANOVA F-statistic avalia, para cada feature, se existe diferença significativa de média entre o grupo da classe-alvo (ex: MI = 1) e os demais (MI = 0). Uma F-statistic elevada indica que a feature separa bem aquela classe das demais.

O esquema **one-vs-rest** é aplicado para cada uma das 5 superclasses, e a **F máxima** entre as classes é usada como indicador de relevância geral da feature.

In [ ]:
superclasses = ['NORM', 'MI', 'CD', 'STTC', 'HYP']

df_tr = df_scaled[df_scaled['strat_fold'].isin(FOLDS_TREINO)].copy()
feature_cols = [c for c in df_scaled.columns if c not in META_COLS]

f_por_classe = {}
for sc in superclasses:
    # Rótulo binário: 1 se a superclasse está na lista, 0 caso contrário
    y = df_tr['diagnostic_superclass'].apply(
        lambda x: 1 if sc in x else 0
    )
    if y.nunique() < 2:
        f_por_classe[sc] = np.zeros(len(feature_cols))
        continue
    F, _ = f_classif(df_tr[feature_cols], y)
    F    = np.nan_to_num(F, nan=0.0)
    f_por_classe[sc] = F

df_anova = pd.DataFrame(f_por_classe, index=feature_cols)
df_anova['F_max'] = df_anova.max(axis=1)
df_anova['classe_max'] = df_anova[superclasses].idxmax(axis=1)
df_anova = df_anova.sort_values('F_max', ascending=False)

print('Top 15 features por F-statistic máxima:')
display(df_anova[['F_max', 'classe_max'] + superclasses].head(15).round(2))

In [ ]:
# Visualização: Top 30 features — F-statistic máxima
fig, ax = plt.subplots(figsize=(10, 10))

top30 = df_anova.head(30).reset_index()
top30.columns = ['feature'] + list(top30.columns[1:])

palette_cls = {'NORM': '#2ecc71', 'MI': '#e74c3c',
               'CD': '#3498db', 'STTC': '#f39c12', 'HYP': '#9b59b6'}
cores = top30['classe_max'].map(palette_cls)

sns.barplot(data=top30, x='F_max', y='feature',
            palette=cores.tolist(), hue='classe_max',
            dodge=False, legend=True, ax=ax)

ax.set_title('Top 30 Features — ANOVA F-Statistic Máxima (One-vs-Rest, Treino)',
             fontsize=12)
ax.set_xlabel('F-Statistic Máxima')
ax.set_ylabel('Feature')
ax.legend(title='Classe discriminada', fontsize=9)

plt.tight_layout()
plt.savefig(FIGS_DIR / 'anova_top30_features.png', dpi=150, bbox_inches='tight')
plt.show()

**Comentários sobre a subseção 4.2 — ANOVA F-statistic Top 30:**

> *Preencha esta célula após executar o bloco acima. Orientações específicas para o que analisar:*

- **Domínios dominantes no Top 30:** anote quais prefixos de features aparecem mais no ranking (`time_`, `freq_`, `morph_`, `wavelet_`, `nonlin_`, `ratio_`, `norm_baseline_`, `delta_`). O resultado esperado, com base na literatura de ECG, é que features morfológicas (amplitude R, largura QRS) e espectrais (razão QRS/total) dominem o ranking, pois refletem diretamente as diferenças estruturais entre as patologias. Se features não-lineares aparecerem no Top 30, isso indica que a complexidade da dinâmica cardíaca é discriminativa — o que seria um resultado interessante para discutir.

- **Presença das features derivadas (E7) no Top 30:** verifique especificamente se `ratio_qrs_pt_II`, `ratio_qrs_total_II`, `norm_baseline_rms_II` ou algum `delta_` aparecem entre as 30 mais discriminativas. Se sim, isso valida concretamente a estratégia de engenharia de features — significa que a combinação de atributos de primeira ordem gerou informação que não estava presente individualmente. Se nenhuma feature derivada aparecer no Top 30, discuta por que isso não invalida a engenharia: features derivadas podem ter F individual menor mas agregar informação complementar importante quando combinadas com outras no PCA ou no classificador.

- **Distribuição por superclasse (cor das barras):** comente a proporção de barras de cada cor. É esperado que **HYP** concentre features de amplitude no topo (RMS, MAV, amplitude R em V5 e V4 — derivações que "enxergam" o ventrículo esquerdo). Para **CD**, features de forma temporal (largura QRS, assimetria QRS) e de baixa frequência do espectro QRS devem ser mais discriminativas — bloqueios de ramo produzem QRS largo que redistribui energia para 5–10 Hz. Para **MI**, features de ST e onda T (morfologia, subbanda A4 da wavelet) devem ter F elevada. Se a distribuição entre classes for muito homogênea (todas as classes com poucas features no Top 30), isso indica que as classes são difíceis de separar individualmente e o classificador precisará combinar muitas features — o que reforça a importância da etapa de Redução de Dimensionalidade.

- **Features com F próxima de zero:** verifique o final do ranking (Bottom 30, que pode ser inspecionado no `df_anova` com `.tail(30)`). Features com F ≈ 0 para todas as classes são candidatas diretas à eliminação no Entregável 9. Se forem muitas (> 20% do total), isso indica que o dataset tem alta redundância ou que algumas features do E6 foram extraídas de forma que não discriminam as patologias do PTB-XL.

### 4.3 Análise de Redundância (Correlação de Pearson)

A análise de redundância é realizada sobre o conjunto de features do Top 50 por F-statistic, que é o subconjunto mais relevante e onde a redundância tem maior impacto prático. Uma matriz de correlação completa sobre todas as features seria computacionalmente custosa e difícil de interpretar visualmente.

In [ ]:
# Subconjunto: Top 50 features por F-statistic máxima
top50_feats = df_anova.head(50).index.tolist()

corr_matrix = df_scaled.loc[mask_treino, top50_feats].corr(method='pearson')

# Identifica pares com |r| > 0.90 (redundância alta)
corr_upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)
pares_redundantes = (
    corr_upper.stack()
    .reset_index()
    .rename(columns={'level_0': 'Feature A', 'level_1': 'Feature B', 0: 'r'})
)
pares_redundantes['|r|'] = pares_redundantes['r'].abs()
pares_redundantes = pares_redundantes[pares_redundantes['|r|'] > 0.90].sort_values('|r|', ascending=False)

print(f'Pares com |r| > 0.90 (Top 50 features): {len(pares_redundantes)}')
if len(pares_redundantes) > 0:
    display(pares_redundantes.head(15).round(4))

In [ ]:
# Heatmap de correlação — Top 50 features
# Rótulos abreviados para legibilidade
labels = [f.replace('_median', '_med').replace('_power', '_pwr')
           .replace('wavelet_', 'wv_').replace('nonlin_', 'nl_')
           .replace('freq_', 'fr_').replace('time_', 't_')
           .replace('morph_', 'm_').replace('hrv_', 'h_')
          for f in top50_feats]

fig, ax = plt.subplots(figsize=(18, 16))
mask_tri = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

sns.heatmap(
    corr_matrix,
    mask=mask_tri,
    annot=False,
    cmap='RdBu_r',
    vmin=-1, vmax=1,
    linewidths=0.2,
    xticklabels=labels,
    yticklabels=labels,
    ax=ax
)
ax.set_title('Matriz de Correlação de Pearson — Top 50 Features (Treino)', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'heatmap_redundancia_top50.png', dpi=150, bbox_inches='tight')
plt.show()

**Comentários sobre a subseção 4.3 — Análise de Redundância (Correlação de Pearson, Top 50):**

> *Preencha esta célula após executar o bloco acima. Orientações específicas para o que analisar:*

- **Blocos de alta correlação intradomínio (esperados):** features do mesmo tipo calculadas em derivações vizinhas anatomicamente devem formar blocos visíveis no heatmap. As derivações do plano frontal (I, II, III, aVL, aVR, aVF) são dependentes entre si por construção (as derivações de Einthoven satisfazem DI + DIII = DII), de modo que features como `time_rms_I`, `time_rms_II` e `time_rms_III` devem apresentar correlações altas entre si. Isso não é um problema de engenharia — é uma propriedade física do ECG de 12 derivações. Comente se esse padrão é visível no heatmap.

- **Pares matematicamente equivalentes (correlação ≈ 1,0):** dois pares têm equivalência algébrica direta e devem aparecer com r muito próximo de 1: **(a)** `hrv_rmssd` e `nonlin_sd1` — por definição, SD1 = RMSSD / √2, portanto r = 1,0 entre eles; **(b)** `time_rms_II` e `norm_baseline_rms_II` — a segunda é a primeira dividida por uma constante (mediana do NORM no treino), então r = 1,0. Esses pares devem ser identificados na tabela de pares redundantes e um deles deve ser eliminado no Entregável 9, pois manter ambos não adiciona informação e ocupa dimensionalidade desnecessária.

- **Correlação entre features derivadas e suas origens:** verifique a correlação entre `ratio_qrs_pt_II` e os seus dois componentes `freq_qrs_power_II` e `freq_pt_power_II` separadamente. Uma razão tem correlação positiva com o numerador e negativa com o denominador — mas se o denominador (banda PT) tiver variância muito menor que o numerador (banda QRS), a razão será dominada pelo numerador e terá r alto com ele. Discuta se `ratio_qrs_pt_II` apresenta informação genuinamente nova ou se é redundante com `freq_qrs_power_II`.

- **Features de baixa correlação com todas as outras (candidatas a manter):** no heatmap, features que não pertencem a nenhum bloco colorido — cuja linha/coluna tem tonalidade próxima de branco em toda a extensão — são as mais "ortogonais" ao restante do dataset. Essas são candidatas prioritárias a permanecer no dataset final do Entregável 9, independentemente da F-statistic, pois adicionam informação não redundante. Features não-lineares como `nonlin_dfa_alpha` e `nonlin_higuchi_fd` frequentemente se enquadram nesse perfil — discuta se isso ocorre no seu resultado.

- **Implicação para o número de componentes PCA no E8:** o número de pares com |r| > 0,90 indica a redundância estrutural do dataset. Se, por exemplo, forem encontrados 20 pares redundantes em 50 features, a dimensionalidade efetiva do dataset é consideravelmente menor que 50, e o PCA deve capturar ≥ 95% da variância com poucos componentes. Use essa estimativa para formular a hipótese de número de componentes antes de executar o PCA no Entregável 8.

---

## Seção 5 — Validação Final do Dataset Engenhado

Antes de salvar o artefato final, realizamos três verificações rápidas de sanidade: integridade (sem NaNs), distribuição por classe e comparação de dimensões com o dataset de entrada.

In [ ]:
feature_cols_final = [c for c in df_scaled.columns if c not in META_COLS]

# ── 5.1 Integridade ──────────────────────────────────────────────────────────
n_nans = df_scaled[feature_cols_final].isnull().sum().sum()
assert n_nans == 0, f'NaNs encontrados após engenharia: {n_nans}'
print(f'Integridade: sem valores ausentes ({n_nans} NaNs).')

# ── 5.2 Dimensões ────────────────────────────────────────────────────────────
n_orig    = len([c for c in feature_cols_final
                 if not c.startswith(('ratio_', 'norm_baseline_', 'delta_'))])
n_deriv   = len(feature_cols_final) - n_orig

print(f'\nFeatures originais (E6)   : {n_orig}')
print(f'Features derivadas (E7)   : {n_deriv}')
print(f'Total de features          : {len(feature_cols_final)}')
print(f'Registros                  : {len(df_scaled)}')

# ── 5.3 Distribuição por classe e fold ───────────────────────────────────────
print('\nDistribuição por superclasse primária:')
display(df_scaled['label_primary'].value_counts()
        .rename('registros').to_frame()
        .assign(pct=lambda d: (d['registros'] / len(df_scaled) * 100).round(1)))

In [ ]:
# Boxplot de 4 features-chave (uma por domínio) no dataset engenhado
feats_box = []
for col in ['time_rms_II', 'ratio_qrs_pt_II', 'hrv_rmssd', 'nonlin_higuchi_fd']:
    if col in df_scaled.columns:
        feats_box.append(col)

classes_order = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
df_box = df_scaled[df_scaled['label_primary'].isin(classes_order)].copy()

fig, axes = plt.subplots(1, len(feats_box), figsize=(5 * len(feats_box), 6))
if len(feats_box) == 1:
    axes = [axes]

for ax, feat in zip(axes, feats_box):
    sns.boxplot(
        data=df_box, x='label_primary', y=feat,
        order=classes_order, hue='label_primary',
        palette='Spectral', legend=False, ax=ax
    )
    ax.set_title(feat.replace('_', '\n'), fontsize=9)
    ax.set_xlabel('Superclasse')
    ax.set_ylabel('Valor Escalado (RobustScaler)')

plt.suptitle('Features-Chave Pós-Engenharia — Separabilidade por Classe (Treino)',
             fontsize=12)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'boxplot_features_engenhadas_por_classe.png', dpi=150, bbox_inches='tight')
plt.show()

**Comentários sobre a Seção 5 — Validação Final: Boxplots e Distribuição de Classes:**

> *Preencha esta célula após executar o bloco acima. Orientações específicas para o que analisar:*

- **`time_rms_II` escalado — separabilidade de HYP:** após o RobustScaler, os valores de HYP devem estar consistentemente acima de 0 nessa feature (mediana positiva), enquanto NORM deve estar centrado próximo de 0 por construção. Comente se os IQRs de HYP e NORM se sobrepõem muito — se a sobreposição for grande, significa que o RMS em DII isoladamente não é suficiente para discriminar HYP, e que features adicionais (ex: amplitude R em V5, que é a derivação do critério de Sokolow-Lyon) serão necessárias no classificador.

- **`ratio_qrs_pt_II` — validação da engenharia de features:** este é o resultado mais importante desta seção para avaliar se a Seção 2 deste entregável agregou valor. Compare visualmente a separação entre classes nessa feature com o que seria esperado nas features originais `freq_qrs_power_II` e `freq_pt_power_II` separadamente (que podem ser inspecionadas rapidamente com `df_scaled.boxplot('freq_qrs_power_II', by='label_primary')`). Se `ratio_qrs_pt_II` mostrar melhor separação — especialmente entre CD e NORM, onde o QRS alargado redistribui energia — a engenharia de features foi efetiva. Discuta esse resultado explicitamente.

- **`hrv_rmssd` escalado — variabilidade autonômica por classe:** comente a posição relativa das medianas. CD, que inclui distúrbios de condução como bloqueio AV e AFIB, pode apresentar padrão bimodal (alguns registros com RMSSD muito alto por ritmo irregular, outros com RMSSD baixo por bradicardia severa) — isso apareceria como IQR muito largo para CD no boxplot. NORM deve ter distribuição mais concentrada. HYP, dependendo da presença de arritmias associadas, pode ter comportamento variável — discuta o que é observado.

- **`nonlin_higuchi_fd` escalado — complexidade do sinal:** a dimensão fractal é uma das features mais interpretáveis clinicamente neste grupo. Coração saudável (NORM) deve apresentar complexidade intermediária (Higuchi FD em torno de 1,4–1,6 antes do escalonamento, portanto próximo de 0 após). Patologias que regularizam o ritmo (alguns bloqueios em CD, MI com disfunção autonômica grave) devem reduzir a FD — aparecem como valores negativos no eixo escalado. Patologias que aumentam a irregularidade (AFIB) devem elevar a FD. Discuta se o padrão observado é consistente com esse raciocínio.

- **Balanceamento das classes:** o PTB-XL é um dataset multirrótulo com distribuição naturalmente desbalanceada — NORM e MI têm muito mais registros que HYP. Discuta se o desbalanceamento observado é expressivo o suficiente para exigir estratégias de reamostragem (SMOTE, class weights) no Entregável 10, ou se a proporção entre as classes permite treinamento direto com ajuste de pesos no classificador.

---

## Seção 6 — Persistência do Dataset Final e Síntese

### 6.1 Salvamento dos Artefatos

In [ ]:
# Dataset final engenhado e normalizado
path_parquet = OUT_DIR / 'features_engineered.parquet'
path_csv     = OUT_DIR / 'features_engineered_sample.csv'
path_catalog = OUT_DIR / 'feature_catalog_e7.csv'

df_scaled.to_parquet(str(path_parquet), index=True)
df_scaled.head(500).to_csv(str(path_csv), index=True)

# Catálogo de features com origem e domínio
def get_domain(col):
    if col.startswith('time_')          : return 'Tempo'
    if col.startswith('freq_')          : return 'Frequência'
    if col.startswith('hrv_')           : return 'HRV'
    if col.startswith('morph_')         : return 'Morfologia'
    if col.startswith('wavelet_')       : return 'Wavelet'
    if col.startswith('nonlin_')        : return 'Não-Linear'
    if col.startswith('ratio_')         : return 'Derivada — Razão Espectral'
    if col.startswith('norm_baseline_') : return 'Derivada — Baseline NORM'
    if col.startswith('delta_')         : return 'Derivada — Delta Inter-Deriv.'
    return 'Outro'

cat_rows = []
for col in feature_cols_final:
    cat_rows.append({
        'feature' : col,
        'dominio' : get_domain(col),
        'F_max'   : round(df_anova.loc[col, 'F_max'], 3) if col in df_anova.index else np.nan,
        'classe_max': df_anova.loc[col, 'classe_max'] if col in df_anova.index else '',
        'media_treino': round(float(df_scaled.loc[mask_treino, col].mean()), 4),
        'std_treino'  : round(float(df_scaled.loc[mask_treino, col].std()),  4),
    })

df_catalog = pd.DataFrame(cat_rows).sort_values('F_max', ascending=False)
df_catalog.to_csv(str(path_catalog), index=False)

print('Artefatos salvos:')
print(f'  features_engineered.parquet  — dataset completo ({df_scaled.shape})')
print(f'  features_engineered_sample.csv — primeiras 500 linhas')
print(f'  feature_catalog_e7.csv        — catálogo com domínio e F-statistic')
print(f'  scaler_robust.pkl             — parâmetros do RobustScaler')

### 6.2 Síntese e Conexão com o Entregável 8

#### O que foi feito neste entregável

| Etapa | Resultado |
|---|---|
| Remoção de features quasi-constantes | Features sem variabilidade eliminadas |
| Remoção de features com NaN > 5% | Features não-lineares com cobertura insuficiente eliminadas |
| Imputação por mediana da superclasse (treino-restrita) | NaNs residuais tratados sem data leakage |
| Razões espectrais (QRS/PT, QRS/Total × 12 derivações) | Features adimensionais invariantes à escala de amplitude |
| Normalização por baseline NORM | Features relativas ao comportamento cardíaco normal |
| Deltas morfológicos inter-derivações | Gradientes elétricos entre regiões cardíacas opostas |
| RobustScaler (fit no treino) | Todas as features na mesma escala, robusta a outliers |
| ANOVA F-statistic (one-vs-rest) | Ranking de relevância por classe, Top 30 visualizado |
| Análise de redundância (correlação de Pearson, Top 50) | Pares com |r| > 0,90 identificados para remoção futura |

#### Limitações

- A análise de redundância foi realizada apenas sobre o Top 50 por F-statistic. Um subconjunto diferente poderia revelar outros pares redundantes.
- Features derivadas baseadas em razões podem ser numericamente instáveis quando o denominador é muito próximo de zero — o termo `+ 1e-10` mitiga isso, mas casos extremos devem ser monitorados.
- A remoção formal das features redundantes identificadas na Seção 4.3 será realizada no Entregável 9, em conjunto com métodos filter (Mutual Information, ReliefF) e embedded (LASSO) para uma seleção estatisticamente fundamentada.

#### Próximos passos — Entregável 8

O arquivo `features_engineered.parquet` é a entrada direta do **Entregável 8 (Redução de Dimensionalidade)**, que aplicará:

- **PCA** com análise de variância acumulada (Scree plot) para identificar o número de componentes que preservam ≥ 95% da variância.
- **ICA** para separação de fontes independentes — potencialmente útil para isolar componentes fisiológicos (variabilidade de ritmo vs. morfologia de onda) de artefatos residuais.
- Visualização 2D dos componentes principais, colorida por superclasse diagnóstica, como validação visual da separabilidade antes da classificação.

In [ ]:
# Verificação final
print('═' * 55)
print('   VERIFICAÇÃO FINAL — ENTREGÁVEL 7')
print('═' * 55)
print(f'  Dataset engenhado        : {df_scaled.shape}')
print(f'  Features originais (E6)  : {n_orig}')
print(f'  Features derivadas (E7)  : {n_deriv}')
print(f'  Valores ausentes         : 0')
print(f'  Scaler                   : RobustScaler (fit treino)')
print(f'  Pares redundantes (|r|>0.90, Top50): {len(pares_redundantes)}')
print()
print('  Artefatos gerados:')
print('    - features_engineered.parquet')
print('    - features_engineered_sample.csv')
print('    - feature_catalog_e7.csv')
print('    - scaler_robust.pkl')
print()
print('  Dataset pronto para o Entregável 8 — Redução de Dimensionalidade.')
print('═' * 55)